In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    os.system('git clone https://github.com/vladlead5/tets.git /content/project')
    os.chdir('/content/project/notebooks')
    os.system('pip install tokenizers transformers sentencepiece -q')

sys.path.append('..')
print('Ready. CWD:', os.getcwd())


# Лабораторная работа: Генеративные модели
## Часть 3: BPE токенизация + Fine-tuning GPT-2

В этом ноутбуке:
1. Обучаем **BPE токенизатор** на нашем корпусе
2. Обучаем модели с BPE токенизацией
3. Делаем **fine-tuning distilgpt2** (предобученной GPT-2)
4. Сравниваем все три токенизации

### Что такое BPE?
Byte Pair Encoding — алгоритм субсловной токенизации:
- Начинаем с символьного словаря
- Итеративно сливаем наиболее частые пары символов/подслов
- Получаем баланс между char-level и word-level
- Современный стандарт: GPT-2, GPT-3, LLaMA используют BPE

In [ ]:
import sys
sys.path.append('..')

import os
import json
import math
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.data.dataset import download_data, load_text, clean_text, train_val_split, TokenTextDataset
from src.tokenizers.bpe_tokenizer import BPETokenizer
from src.models.rnn_model import SimpleRNN
from src.models.lstm_model import LSTMModel
from src.models.bilstm_model import BiLSTMModel
from src.models.transformer_model import GPTModel
from src.training.trainer import train_model
from src.generation.generate import generate_rnn, generate_transformer
from src.evaluation.metrics import compute_perplexity
from src.utils.utils import set_seed, get_device, count_parameters

set_seed(42)
device = get_device()
print(f'Устройство: {device}')

## 1. Загрузка и предобработка данных

In [ ]:
download_data('../data/shakespeare.txt')
text = clean_text(load_text('../data/shakespeare.txt'))[:500_000]
train_text, val_text = train_val_split(text, val_fraction=0.1)
print(f'Train: {len(train_text):,} | Val: {len(val_text):,} символов')

## 2. BPE Токенизация

In [ ]:
BPE_VOCAB_SIZE = 3000

os.makedirs('../outputs/metrics', exist_ok=True)
bpe_tokenizer_path = '../outputs/metrics/bpe_tokenizer.json'

bpe = BPETokenizer(vocab_size=BPE_VOCAB_SIZE)

if os.path.exists(bpe_tokenizer_path):
    print('Загружаем существующий BPE tokenizer...')
    bpe.load(bpe_tokenizer_path)
else:
    print('Обучаем BPE tokenizer...')
    bpe.train(text, save_path=bpe_tokenizer_path)

print(f'BPE vocab_size: {bpe.vocab_size}')

In [ ]:
sample = 'To be, or not to be, that is the question.'
sample_tokens = bpe.show_sample_tokens(sample)

print(f'Оригинал: {sample}')
print(f'\nBPE токены ({len(sample_tokens)}):')
for token, idx in sample_tokens:
    print(f'  "{token}" → {idx}')

encoded = bpe.encode(sample)
decoded = bpe.decode(encoded)
print(f'\nDecoded: {decoded}')

In [ ]:
sample_texts = [text[i:i+200] for i in range(0, 2000, 200)]

char_lengths = [len(t) for t in sample_texts]
bpe_lengths  = [len(bpe.encode(t)) for t in sample_texts]

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(sample_texts))
ax.bar([i - 0.2 for i in x], char_lengths, width=0.4, label='Char tokens', alpha=0.8, color='steelblue')
ax.bar([i + 0.2 for i in x], bpe_lengths,  width=0.4, label='BPE tokens',  alpha=0.8, color='orange')
ax.set_xlabel('Фрагмент текста')
ax.set_ylabel('Количество токенов')
ax.set_title('Char vs BPE: длина последовательностей (одинаковый текст)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/plots/bpe_vs_char_lengths.png', dpi=150)
plt.show()

ratio = sum(char_lengths) / sum(bpe_lengths)
print(f'BPE сжатие относительно char: {ratio:.2f}x (BPE в {ratio:.2f} раза короче)')

In [ ]:
SEQ_LEN = 64
BATCH_SIZE = 64

train_ids = bpe.encode(train_text)
val_ids   = bpe.encode(val_text)

train_dataset = TokenTextDataset(train_ids, SEQ_LEN)
val_dataset   = TokenTextDataset(val_ids, SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

print(f'Train: {len(train_ids):,} BPE-токенов → {len(train_dataset):,} примеров')
print(f'Val:   {len(val_ids):,} BPE-токенов → {len(val_dataset):,} примеров')

## 3. Обучение моделей с BPE

In [ ]:
VOCAB_SIZE = bpe.vocab_size
NUM_EPOCHS = 5

models_config = {
    'SimpleRNN': {
        'class': SimpleRNN,
        'kwargs': dict(vocab_size=VOCAB_SIZE, embed_dim=128, hidden_size=256, num_layers=1, dropout=0.3),
        'type': 'rnn', 'lr': 1e-3,
    },
    'LSTM-1layer': {
        'class': LSTMModel,
        'kwargs': dict(vocab_size=VOCAB_SIZE, embed_dim=128, hidden_size=256, num_layers=1, dropout=0.3),
        'type': 'rnn', 'lr': 1e-3,
    },
    'LSTM-2layer': {
        'class': LSTMModel,
        'kwargs': dict(vocab_size=VOCAB_SIZE, embed_dim=128, hidden_size=256, num_layers=1, dropout=0.3),
        'type': 'rnn', 'lr': 1e-3,
    },
    'BiLSTM': {
        'class': BiLSTMModel,
        'kwargs': dict(vocab_size=VOCAB_SIZE, embed_dim=128, hidden_size=256, num_layers=1, dropout=0.3),
        'type': 'rnn', 'lr': 1e-3,
    },
    'GPT': {
        'class': GPTModel,
        'kwargs': dict(vocab_size=VOCAB_SIZE, embed_dim=256, num_heads=4, num_layers=4, max_seq_len=SEQ_LEN, dropout=0.1),
        'type': 'transformer', 'lr': 3e-4,
    },
}

print(f'Vocab size: {VOCAB_SIZE}')
print(f"{'Модель':20s} | {'Параметры':>12s}")
print('-'*36)
for name, cfg in models_config.items():
    m = cfg['class'](**cfg['kwargs'])
    print(f'{name:20s} | {count_parameters(m):>12,}')

In [ ]:
all_histories = {}

for model_name, cfg in models_config.items():
    set_seed(42)
    model = cfg['class'](**cfg['kwargs'])
    history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        model_name=f'bpe_{model_name}',
        model_type=cfg['type'],
        num_epochs=NUM_EPOCHS,
        lr=cfg['lr'],
        checkpoint_dir='../outputs/checkpoints',
        device=device,
    )
    all_histories[model_name] = history

print('Обучение завершено!')

## 4. Fine-tuning distilgpt2 (бонус — на отличную оценку)

In [ ]:
from src.models.gpt_finetune import finetune_gpt2, generate_gpt2, load_pretrained_gpt2

print('Запускаем fine-tuning distilgpt2...')
print('(Занимает несколько минут на CPU, 30-60 сек на GPU)')

finetune_text = text[:200_000]

gpt2_model, gpt2_tokenizer, gpt2_history = finetune_gpt2(
    text=finetune_text,
    output_dir='../outputs/checkpoints',
    model_name='distilgpt2',
    seq_len=128,
    batch_size=4,
    num_epochs=3,
    lr=5e-5,
    device=device,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, len(gpt2_history['train_loss']) + 1)
axes[0].plot(epochs, gpt2_history['train_loss'], 'b-o', label='Train')
axes[0].plot(epochs, gpt2_history['val_loss'], 'r-o', label='Val')
axes[0].set_title('distilgpt2 Fine-tuning Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, gpt2_history['val_ppl'], 'g-o')
axes[1].set_title('distilgpt2 Val Perplexity')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('PPL')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/plots/gpt2_finetune_curves.png', dpi=150)
plt.show()

In [ ]:
prompts = [
    'To be, or not to be',
    'KING HENRY: My lord',
    'What light through yonder',
]

print('Генерации fine-tuned distilgpt2:\n')
gpt2_generations = {}

for prompt in prompts:
    gen = generate_gpt2(
        gpt2_model, gpt2_tokenizer, prompt,
        max_new_tokens=100, temperature=0.8, top_k=40, device=device
    )
    gpt2_generations[prompt] = gen
    print(f'[Prompt]: {prompt}')
    print(f'[Generated]:')
    print(gen)
    print()

In [ ]:
if 'GPT' in models_config:
    from src.generation.generate import generate_transformer
    
    from src.tokenizers.char_tokenizer import CharTokenizer
    char_tok = CharTokenizer()
    char_tok_path = '../outputs/metrics/char_tokenizer.json'
    if os.path.exists(char_tok_path):
        char_tok.load(char_tok_path)
        our_gpt = GPTModel(vocab_size=char_tok.vocab_size, embed_dim=128, num_heads=4, 
                           num_layers=4, max_seq_len=100, dropout=0.1)
        ckpt_path = '../outputs/checkpoints/char_GPT_best.pt'
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location=device)
            our_gpt.load_state_dict(ckpt['model_state_dict'])
            our_gpt = our_gpt.to(device)
            
            prompt = 'To be, or not to be'
            our_gen = generate_transformer(our_gpt, char_tok, prompt, max_new_tokens=100,
                                           strategy='temperature', device=device)
            
            print('СРАВНЕНИЕ:\n')
            print(f'[Наш GPT (char-level)]:')  
            print(our_gen[:300])
            print()
            print(f'[Fine-tuned distilgpt2]:')
            print(gpt2_generations.get(prompt, 'N/A')[:300])

## 5. Итоговое сравнение всех токенизаций

In [ ]:
summary_rows = []

for tokenization in ['char', 'word', 'bpe']:
    csv_path = f'../outputs/metrics/{tokenization}_comparison.csv'
    if os.path.exists(csv_path):
        df_tok = pd.read_csv(csv_path)
        for _, row in df_tok.iterrows():
            summary_rows.append({
                'Токенизация': tokenization.upper(),
                'Модель': row['Модель'],
                'Best Val Loss': row['Best Val Loss'],
                'Best Val PPL': row['Best Val PPL'],
            })

if gpt2_history:
    best_gpt2_loss = min(gpt2_history['val_loss'])
    summary_rows.append({
        'Токенизация': 'GPT-2 tokenizer',
        'Модель': 'distilgpt2 (fine-tuned)',
        'Best Val Loss': f'{best_gpt2_loss:.4f}',
        'Best Val PPL': f'{compute_perplexity(best_gpt2_loss):.1f}',
    })

if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    print('ИТОГОВОЕ СРАВНЕНИЕ:')
    print(summary_df.to_string(index=False))
    summary_df.to_csv('../outputs/metrics/full_comparison.csv', index=False)
else:
    print('Запустите CharTokenization.ipynb и WordTokenization.ipynb сначала.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

for (name, hist), color in zip(all_histories.items(), colors):
    epochs = range(1, len(hist['val_loss']) + 1)
    axes[0].plot(epochs, hist['train_loss'], '--', color=color, alpha=0.7)
    axes[0].plot(epochs, hist['val_loss'], '-', color=color, label=name, linewidth=2)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Кривые обучения (BPE)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

for (name, hist), color in zip(all_histories.items(), colors):
    epochs = range(1, len(hist['val_ppl']) + 1)
    axes[1].plot(epochs, hist['val_ppl'], '-o', color=color, label=name, markersize=4)

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('PPL')
axes[1].set_title('Val Perplexity (BPE)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/plots/bpe_training_curves.png', dpi=150)
plt.show()

In [ ]:
bpe_gens = {}
for model_name, cfg in models_config.items():
    model = cfg['class'](**cfg['kwargs'])
    path = f'../outputs/checkpoints/bpe_{model_name}_best.pt'
    if os.path.exists(path):
        ckpt = torch.load(path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device)
    
    prompt = 'to be or not'
    if cfg['type'] == 'transformer':
        gen = generate_transformer(model, bpe, prompt, max_new_tokens=50,
                                   strategy='temperature', device=device)
    else:
        gen = generate_rnn(model, bpe, prompt, max_new_tokens=50,
                           strategy='temperature', device=device)
    bpe_gens[model_name] = gen

with open('../outputs/generations/bpe_generations.txt', 'w', encoding='utf-8') as f:
    f.write('ГЕНЕРАЦИИ (BPE токенизация)\n' + '='*60 + '\n\n')
    for name, gen_text in bpe_gens.items():
        f.write(f'[{name}]:\n{gen_text}\n\n')
    f.write('\n--- Fine-tuned distilgpt2 ---\n')
    for prompt, gen_text in gpt2_generations.items():
        f.write(f'\n[Prompt: {prompt}]:\n{gen_text}\n')

print('Все генерации сохранены!')

## 6. Итоговые выводы

### BPE vs Char vs Word:

| Токенизация | Vocab | Длина seq | Семантика | OOV | Популярность |
|-------------|-------|-----------|-----------|-----|-------------|
| Char        | ~65   | Длинная   | Низкая    | Нет | Историческая |
| Word        | 5–50K | Короткая  | Высокая   | Есть| Устаревшая  |
| BPE         | 2–50K | Средняя   | Высокая   | Нет | Современная |

### Fine-tuning vs обучение с нуля:

| Аспект | С нуля | Fine-tuning |
|--------|--------|-------------|
| Время обучения | Долго | Быстро |
| Данных нужно | Много | Мало |
| Начальная точка | Случайная | Предобученные веса |
| Итоговое качество | Хорошее | Отличное |

**Вывод**: Fine-tuning предобученных моделей даёт значительно лучшие результаты с меньшими затратами вычислительных ресурсов. Это основной подход в современном NLP.

### Лучшая конфигурация:
**BPE + GPT-like Transformer** — оптимальный баланс для задачи языкового моделирования.

**Fine-tuned distilgpt2** — наилучшее качество генерации при минимальном времени обучения.